# MovieLens — Stage-1 baselines (Random / BCE / InfoNCE / AdvInfoNCE / MyLoss) — Hit@20 & NDCG@20, all-ranking

Adapted from `yahoo-earlystopping.ipynb`. **Backbone, losses, evaluation metric, optimizer/schedule, early-stopping
logic, and multi-seed aggregation are all unchanged.** The only thing that differs is how the data is loaded and split:

* **Data**: MovieLens (`movies.dat` / `users.dat` / `ratings.dat`), loaded with real user features (gender/age-bucket/occupation
  one-hot, 29-dim) and real movie features (genre multi-hot). This replaces Yahoo!R3's zero-width feature tensors
  — the architecture is feature-dim agnostic, so nothing else changes.
* **Split — leave-one-out per user, by timestamp (old → new)**:
  * every user's **last** interaction (latest timestamp) → **test**
  * every user's **second-to-last** interaction → **validation**
  * every older interaction → **train**

  This replaces Yahoo!R3's fixed train file (`user.txt`, used 100% intact) + uneven MAR split (`random.txt` →
  20% val / 80% test). There is no separate MNAR/MAR distinction in MovieLens, so the leave-one-out-by-time
  protocol (standard for MovieLens sequential/implicit recommendation) is used instead.
* Users with fewer than 2 interactions can't contribute to `train`; users with fewer than 3 can't contribute to
  both `train` and `val`. This is expected and handled automatically by the split.

Everything else — `EMBED_DIM`, `ID_EMB_DIM`, `HIDDEN`, `BATCH_SIZE`, `MAX_EPOCHS`, `PATIENCE`, `LR`,
`WEIGHT_DECAY`, `GRAD_CLIP`, `TEMPERATURE`, the AdvInfoNCE adversary schedule, `SEEDS = [42, 43, 44]`,
`HIT_K = 20` — is copied over unchanged from the Yahoo notebook.


In [1]:
import numpy as np
import pandas as pd
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/datasets/sherinclaudia/movielens/users.dat
/kaggle/input/datasets/sherinclaudia/movielens/ratings.dat
/kaggle/input/datasets/sherinclaudia/movielens/movies.dat


In [2]:
"""
MovieLens Stage-1 baselines: Random / BCE / InfoNCE / AdvInfoNCE / MyLoss (SC-InfoNCE)

Stage-1 ONLY, two metrics: Hit@20 and NDCG@20, evaluated with the all-ranking
strategy (rank every unseen movie for each test user), matching the
AdvInfoNCE paper's evaluation protocol.

Adapted from the Yahoo!R3 early-stopping notebook. Backbone / losses / eval /
optimizer-schedule / early-stopping / multi-seed aggregation are unchanged.
Only the data loading and train/val/test split differ:

  - MovieLens is loaded with real user features (gender, age-bucket,
    occupation one-hot -> 29-dim) and real movie features (genre multi-hot),
    replacing Yahoo!R3's zero-width feature tensors.
  - Split is leave-one-out per user, ordered by timestamp (old -> new):
      last interaction of every user          -> TEST
      second-to-last interaction of every user -> VAL (early stopping)
      every older interaction                  -> TRAIN
    replacing Yahoo!R3's fixed user.txt train file + uneven random.txt
    val/test split.
  - Every trainable baseline is early-stopped on val_ground_truth
    (all-ranking HR@K, monitored every epoch, patience = PATIENCE, hard cap
    MAX_EPOCHS); the best-validation-epoch encoder weights are restored
    before computing the embeddings that get scored on test_ground_truth.
  - SEEDS: [42, 43, 44].
"""

import os, math, random, time, copy
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

POS_THRESHOLD = 4      # rating >= 4 is a positive interaction (paper convention)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# 1. DATA / SPLIT
#    Leave-one-out, per user, by timestamp (old -> new):
#      - last interaction of every user           -> TEST
#      - second-to-last interaction of every user -> VAL (early stopping)
#      - every older interaction                  -> TRAIN
#    Computed once, shared by every baseline and every seed.
# ═══════════════════════════════════════════════════════════════════════
def load_movielens():
    movies = pd.read_csv(
        "/kaggle/input/datasets/sherinclaudia/movielens/movies.dat",
        sep="::", engine="python", encoding="latin1",
        names=["movie_id", "title", "genres"])
    users = pd.read_csv(
        "/kaggle/input/datasets/sherinclaudia/movielens/users.dat",
        sep="::", engine="python", encoding="latin1",
        names=["user_id", "gender", "age", "occupation", "zip_code"])
    ratings = pd.read_csv(
        "/kaggle/input/datasets/sherinclaudia/movielens/ratings.dat",
        sep="::", engine="python", encoding="latin1",
        names=["user_id", "movie_id", "rating", "timestamp"])

    sorted_mids = sorted(movies["movie_id"].unique())
    movie_to_idx = {mid: i for i, mid in enumerate(sorted_mids)}
    movies["movie_idx"] = movies["movie_id"].map(movie_to_idx)

    user_to_idx = {uid: uid - 1 for uid in users["user_id"]}
    users["user_idx"] = users["user_id"].map(user_to_idx)

    num_movies = len(movie_to_idx)
    num_users = len(user_to_idx)

    ratings["user_idx"] = ratings["user_id"].map(user_to_idx)
    ratings["movie_idx"] = ratings["movie_id"].map(movie_to_idx)
    ratings = ratings.dropna(subset=["user_idx", "movie_idx"])
    ratings["user_idx"] = ratings["user_idx"].astype(int)
    ratings["movie_idx"] = ratings["movie_idx"].astype(int)
    ratings = ratings.sort_values(["user_idx", "timestamp"])

    all_genres = sorted({g for gs in movies["genres"] for g in gs.split("|")})
    genre_to_i = {g: i for i, g in enumerate(all_genres)}
    NUM_GENRES = len(all_genres)

    def genres_to_vec(g_str):
        v = np.zeros(NUM_GENRES, dtype=np.float32)
        for g in g_str.split("|"):
            if g in genre_to_i:
                v[genre_to_i[g]] = 1.
        return v

    movies["genre_vec"] = movies["genres"].apply(genres_to_vec)

    AGE_BUCKETS = {1: 0, 18: 1, 25: 2, 35: 3, 45: 4, 50: 5, 56: 6}

    def user_feat_vec(row):
        v = np.zeros(29, dtype=np.float32)
        v[0] = 1. if row["gender"] == "M" else 0.
        v[1 + AGE_BUCKETS.get(row["age"], 0)] = 1.
        v[8 + int(row["occupation"])] = 1.
        return v

    users["feat_vec"] = users.apply(user_feat_vec, axis=1)

    user_feat_tensor = torch.tensor(
        np.stack(users.sort_values("user_idx")["feat_vec"].values), dtype=torch.float32)
    movie_feat_tensor = torch.tensor(
        np.stack(movies.sort_values("movie_idx")["genre_vec"].values), dtype=torch.float32)

    return dict(movies=movies, users=users, ratings=ratings,
                num_users=num_users, num_movies=num_movies,
                user_feat_tensor=user_feat_tensor, movie_feat_tensor=movie_feat_tensor,
                NUM_GENRES=NUM_GENRES)


def leave_one_out_split(ratings):
    """`ratings` should already be sorted ascending by (user_idx, timestamp)
    (load_movielens already does this, but we re-sort defensively).

    Per-user leave-one-out split, oldest -> newest:
      rank_from_end == 0 (latest interaction)          -> test
      rank_from_end == 1 (second-latest interaction)   -> val
      rank_from_end >= 2 (every older interaction)     -> train

    Users with only 1 interaction contribute to test only (no val/train
    rows); users with only 2 contribute to val/test only (no train rows).
    This is the standard leave-one-out protocol and is expected/handled
    automatically everywhere downstream (ground truth, r_lookup, user_seen).
    """
    r = ratings.sort_values(["user_idx", "timestamp"]).reset_index(drop=True)
    rank_from_end = r.groupby("user_idx").cumcount(ascending=False)
    test_df = r[rank_from_end == 0].reset_index(drop=True)
    val_df = r[rank_from_end == 1].reset_index(drop=True)
    train_df = r[rank_from_end >= 2].reset_index(drop=True)
    return train_df, val_df, test_df


def make_ground_truth(split_df, pos_threshold=POS_THRESHOLD):
    """uid -> set of movie_idx with rating >= pos_threshold. Works for either
    val_df or test_df. Users with zero qualifying positives are dropped."""
    pos_df = split_df[split_df["rating"] >= pos_threshold]
    ground_truth = defaultdict(set)
    for row in pos_df.itertuples():
        ground_truth[row.user_idx].add(row.movie_idx)
    return dict(ground_truth)


def make_r_lookup(train_df, num_users, num_movies):
    r_lookup = torch.zeros(num_users, num_movies, dtype=torch.float32)
    uids_all = torch.tensor(train_df["user_idx"].values, dtype=torch.long)
    vids_all = torch.tensor(train_df["movie_idx"].values, dtype=torch.long)
    rats_all = torch.tensor(train_df["rating"].values, dtype=torch.float32)
    r_lookup[uids_all, vids_all] = rats_all
    return r_lookup.to(DEVICE)


def make_user_seen(train_df):
    """All train interactions (any rating) get masked out at eval time --
    for BOTH val and test, since these are interactions the model was
    actually trained on."""
    user_seen = defaultdict(set)
    for row in train_df.itertuples():
        user_seen[row.user_idx].add(row.movie_idx)
    return user_seen


class CLIPDataset(Dataset):
    """rating >= POS_THRESHOLD only; returns uid+vid for ID embeddings + raw features."""
    def __init__(self, train_df, user_feat_tensor, movie_feat_tensor, pos_threshold=POS_THRESHOLD):
        pos_df = train_df[train_df["rating"] >= pos_threshold].copy()
        self.user_idxs = pos_df["user_idx"].values.astype(int)
        self.movie_idxs = pos_df["movie_idx"].values.astype(int)
        self.user_feats = user_feat_tensor
        self.movie_feats = movie_feat_tensor

    def __len__(self):
        return len(self.user_idxs)

    def __getitem__(self, idx):
        uid = int(self.user_idxs[idx])
        vid = int(self.movie_idxs[idx])
        return {"uid": uid, "vid": vid,
                "user_feats": self.user_feats[uid],
                "movie_feats": self.movie_feats[vid]}


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# 2. SHARED ARCHITECTURE — identical across BCE / InfoNCE / AdvInfoNCE / MyLoss
#    (unchanged from the Yahoo notebook)
# ═══════════════════════════════════════════════════════════════════════
EMBED_DIM = 256
ID_EMB_DIM = 64
HIDDEN = (256, 128)          # -> 3 linear layers total per tower (2 hidden + 1 out)


def make_tower(input_dim, hidden=HIDDEN, out_dim=EMBED_DIM):
    layers, in_d = [], input_dim
    for h in hidden:
        layers += [nn.Linear(in_d, h), nn.LayerNorm(h), nn.ReLU()]
        in_d = h
    layers.append(nn.Linear(in_d, out_dim))
    return nn.Sequential(*layers)


class TwoTowerEncoder(nn.Module):
    """The ONE backbone every trainable baseline shares. No loss logic here."""
    def __init__(self, num_users, num_movies, user_feat_dim, movie_feat_dim,
                 id_emb_dim=ID_EMB_DIM, hidden=HIDDEN, out_dim=EMBED_DIM):
        super().__init__()
        self.user_id_emb = nn.Embedding(num_users, id_emb_dim)
        self.movie_id_emb = nn.Embedding(num_movies, id_emb_dim)
        self.user_tower = make_tower(user_feat_dim + id_emb_dim, hidden, out_dim)
        self.movie_tower = make_tower(movie_feat_dim + id_emb_dim, hidden, out_dim)

    def encode_user(self, user_feats, uid):
        id_e = self.user_id_emb(uid)
        return F.normalize(self.user_tower(torch.cat([user_feats, id_e], dim=-1)), dim=-1)

    def encode_movie(self, movie_feats, vid):
        id_e = self.movie_id_emb(vid)
        return F.normalize(self.movie_tower(torch.cat([movie_feats, id_e], dim=-1)), dim=-1)


class HardnessModel(nn.Module):
    """AdvInfoNCE-embed adversary: separate, smaller embedding table."""
    def __init__(self, num_users, num_movies, dim=ID_EMB_DIM):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, dim)
        self.movie_emb = nn.Embedding(num_movies, dim)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.movie_emb.weight, std=0.01)

    def raw_scores(self, uid, vid):
        """Full in-batch score matrix g(u_a, i_b) for a batch, shape (B,B)."""
        u = self.user_emb(uid)     # (B, d)
        v = self.movie_emb(vid)    # (B, d)
        return u @ v.T             # (B, B)


In [5]:
# ═══════════════════════════════════════════════════════════════════════
# 3. LOSS FUNCTIONS
#    Only the loss function is allowed to differ across baselines; everything
#    else is shared, per the fairness rationale in the module docstring.
#    (unchanged from the Yahoo notebook)
# ═══════════════════════════════════════════════════════════════════════
def infonce_loss(u_emb, v_emb, temperature=0.07):
    """Symmetric in-batch InfoNCE."""
    logits = (u_emb @ v_emb.T) / temperature
    targets = torch.arange(logits.size(0), device=logits.device)
    return (F.cross_entropy(logits, targets) + F.cross_entropy(logits.T, targets)) / 2


def weighted_clip_loss(u_emb, v_emb, uid, vid, r_lookup, temperature=0.07):
    """MyLoss / SC-InfoNCE. W = raw rating / 5.0 (1-5 scale)."""
    B = u_emb.size(0)
    uid = uid.clamp(0, r_lookup.size(0) - 1)
    vid = vid.clamp(0, r_lookup.size(1) - 1)

    logits = (u_emb @ v_emb.T) / temperature
    W = r_lookup[uid[:, None], vid[None, :]] / 5.0

    EPS = 1e-2
    Wc = torch.clamp(W, min=EPS, max=1.0)

    # Reciprocal weighting for negatives: 1/W
    log_weights = torch.log(1.0 / Wc)

    # Positive pairs keep the original positive weighting: W
    diag_mask = torch.eye(B, device=logits.device, dtype=torch.bool)
    log_weights[diag_mask] = torch.log(
        torch.clamp(W[diag_mask], min=EPS)
    )

    weighted_logits = logits + log_weights
    targets = torch.arange(B, device=logits.device)

    return (
        F.cross_entropy(weighted_logits, targets) +
        F.cross_entropy(weighted_logits.T, targets)
    ) / 2


def advinfonce_loss(u_emb, v_emb, hardness_model, uid, vid, temperature=0.07, K=64.0):
    """In-batch AdvInfoNCE (AdvInfoNCE-embed), symmetric."""
    B = u_emb.size(0)
    logits = (u_emb @ v_emb.T) / temperature           # (B, B), s(u_a, i_b)
    diag_mask = torch.eye(B, device=logits.device, dtype=torch.bool)

    raw = hardness_model.raw_scores(uid, vid)           # (B, B)
    raw_masked = raw.masked_fill(diag_mask, float("-inf"))
    log_p = F.log_softmax(raw_masked, dim=-1)           # p(j|u,i), zero mass on diagonal

    log_K = math.log(K)
    log_weighted_neg = log_K + log_p + logits
    log_weighted_neg = log_weighted_neg.masked_fill(diag_mask, float("-inf"))

    s_pos = logits[diag_mask].view(B)
    all_terms = torch.cat([s_pos.unsqueeze(1), log_weighted_neg], dim=1)
    log_denom = torch.logsumexp(all_terms, dim=1)
    loss_row = -(s_pos - log_denom)

    logits_T = logits.T
    raw_T = raw.T
    raw_T_masked = raw_T.masked_fill(diag_mask, float("-inf"))
    log_p_T = F.log_softmax(raw_T_masked, dim=-1)
    log_weighted_neg_T = (log_K + log_p_T + logits_T).masked_fill(diag_mask, float("-inf"))
    s_pos_T = logits_T[diag_mask].view(B)
    all_terms_T = torch.cat([s_pos_T.unsqueeze(1), log_weighted_neg_T], dim=1)
    log_denom_T = torch.logsumexp(all_terms_T, dim=1)
    loss_col = -(s_pos_T - log_denom_T)

    return (loss_row.mean() + loss_col.mean()) / 2


def bce_loss(u_emb, v_emb, temperature=0.07, pos_weight_val=None):
    """Point-wise BCE baseline over the same in-batch B x B pair matrix.
    F.binary_cross_entropy_with_logits (sigmoid) over every cell of the
    B x B pair matrix, never a softmax/categorical cross-entropy like
    InfoNCE. pos_weight = B - 1 balances the single positive cell per row
    against its B - 1 negative cells (exact negative:positive ratio)."""
    B = u_emb.size(0)
    logits = (u_emb @ v_emb.T) / temperature
    targets = torch.eye(B, device=logits.device)
    pw = torch.tensor(float(B - 1) if pos_weight_val is None else pos_weight_val,
                       device=logits.device)
    return F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pw)


In [6]:
# ═══════════════════════════════════════════════════════════════════════
# 4. EVALUATION — all-ranking, Hit@20 and NDCG@20.
#    Used for BOTH validation (early stopping) and final test scoring.
#      HR@20   = 1{|top20 ∩ ground_truth(u)| > 0}                (binary hit)
#      NDCG@20 = DCG@20 / IDCG@20, relevance=1 for ground-truth positives
#    both averaged over users. (unchanged from the Yahoo notebook)
# ═══════════════════════════════════════════════════════════════════════
HIT_K = 20


def _dcg_at_k_multi(hit_flags, k):
    dcg = 0.0
    for rank, hit in enumerate(hit_flags[:k]):
        if hit:
            dcg += 1.0 / math.log2(rank + 2)
    return dcg


def _idcg_at_k(num_pos, k):
    n = min(num_pos, k)
    return sum(1.0 / math.log2(r + 2) for r in range(n)) if n > 0 else 0.0


def evaluate_retriever_full(user_vecs, movie_vecs, ground_truth, user_seen, K=HIT_K):
    """Full ranking among all unseen movies (no negative sampling). Returns Hit@K and NDCG@K.
    `ground_truth` can be either val_ground_truth or test_ground_truth -- only
    train interactions (`user_seen`) are ever masked out of the candidate pool."""
    U = torch.tensor(user_vecs, dtype=torch.float32).to(DEVICE)
    V = torch.tensor(movie_vecs, dtype=torch.float32).to(DEVICE)
    hits = 0.0
    ndcg = 0.0
    total = 0

    with torch.no_grad():
        for uid, pos_items in ground_truth.items():
            if len(pos_items) == 0:
                continue
            total += 1
            seen = user_seen.get(uid, set())
            scores = (U[uid:uid + 1] @ V.T).squeeze(0).clone()
            if seen:
                scores[torch.tensor(list(seen), dtype=torch.long, device=DEVICE)] = float("-inf")
            ranked = torch.argsort(scores, descending=True).cpu().tolist()

            topk = ranked[:K]
            hit_flags = [1 if item in pos_items else 0 for item in topk]
            hits += 1.0 if any(hit_flags) else 0.0
            ndcg += _dcg_at_k_multi(hit_flags, K) / max(_idcg_at_k(len(pos_items), K), 1e-8)

    return {f"HR@{K}": hits / total, f"NDCG@{K}": ndcg / total}


In [7]:
# ═══════════════════════════════════════════════════════════════════════
# 5. STAGE-1 TRAINING DRIVER — dispatches on loss name, early-stops on
#    validation HR@K, restores the best-validation-epoch encoder weights.
#    (unchanged from the Yahoo notebook)
# ═══════════════════════════════════════════════════════════════════════
BATCH_SIZE = 128
MAX_EPOCHS = 40          # hard cap; baselines rarely need all of it once early stopping is on
PATIENCE = 7             # epochs without val-HR@K improvement before stopping
VAL_METRIC = f"HR@{HIT_K}"   # metric monitored for early stopping / checkpoint selection
LR = 1e-3
WEIGHT_DECAY = 1e-2
GRAD_CLIP = 5.0
TEMPERATURE = 0.07

# AdvInfoNCE-specific
ADV_LR = 5e-5
ADV_T_ADV = 5
ADV_E_ADV = 200
ADV_K = 64.0


def get_full_embeddings(encoder, user_feat_tensor, movie_feat_tensor, num_users, num_movies):
    encoder.eval()
    with torch.no_grad():
        user_vecs = []
        for s in range(0, num_users, 512):
            e = min(s + 512, num_users)
            uf = user_feat_tensor[s:e].to(DEVICE)
            uid = torch.arange(s, e, device=DEVICE)
            user_vecs.append(encoder.encode_user(uf, uid).cpu().numpy())
        user_vecs = np.vstack(user_vecs)

        movie_vecs = []
        for s in range(0, num_movies, 512):
            e = min(s + 512, num_movies)
            mf = movie_feat_tensor[s:e].to(DEVICE)
            vid = torch.arange(s, e, device=DEVICE)
            movie_vecs.append(encoder.encode_movie(mf, vid).cpu().numpy())
        movie_vecs = np.vstack(movie_vecs)
    return user_vecs, movie_vecs


def train_stage1(loss_name, seed, data, train_df, r_lookup, user_seen, val_ground_truth):
    """loss_name in {"Random", "BCE", "InfoNCE", "AdvInfoNCE", "MyLoss"}.

    Trainable baselines use early stopping on val_ground_truth (HR@K), monitored
    every epoch, with patience PATIENCE and a hard cap of MAX_EPOCHS. The encoder
    state with the best validation HR@K is restored before returning.
    """
    set_seed(seed)
    num_users, num_movies = data["num_users"], data["num_movies"]
    user_feat_tensor, movie_feat_tensor = data["user_feat_tensor"], data["movie_feat_tensor"]

    if loss_name == "Random":
        rng = np.random.default_rng(seed)
        user_vecs = rng.normal(size=(num_users, EMBED_DIM)).astype(np.float32)
        movie_vecs = rng.normal(size=(num_movies, EMBED_DIM)).astype(np.float32)
        user_vecs /= np.linalg.norm(user_vecs, axis=1, keepdims=True)
        movie_vecs /= np.linalg.norm(movie_vecs, axis=1, keepdims=True)
        return user_vecs, movie_vecs, None, 0

    encoder = TwoTowerEncoder(num_users, num_movies,
                               user_feat_dim=user_feat_tensor.shape[1],
                               movie_feat_dim=movie_feat_tensor.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(encoder.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS)

    hardness_model = None
    opt_adv = None
    if loss_name == "AdvInfoNCE":
        hardness_model = HardnessModel(num_users, num_movies).to(DEVICE)
        opt_adv = torch.optim.Adam(hardness_model.parameters(), lr=ADV_LR)

    dataset = CLIPDataset(train_df, user_feat_tensor, movie_feat_tensor)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=4, pin_memory=True, drop_last=True)

    global_step = 0
    adv_updates_done = 0

    best_val_metric = -1.0
    best_epoch = 0
    epochs_without_improve = 0
    best_encoder_state = copy.deepcopy(encoder.state_dict())

    for epoch in range(MAX_EPOCHS):
        encoder.train()
        total_loss, steps = 0.0, 0
        t0 = time.time()
        for batch in loader:
            uf = batch["user_feats"].to(DEVICE)
            mf = batch["movie_feats"].to(DEVICE)
            uid = batch["uid"].to(DEVICE)
            vid = batch["vid"].to(DEVICE)

            u = encoder.encode_user(uf, uid)
            v = encoder.encode_movie(mf, vid)

            if loss_name == "InfoNCE":
                loss = infonce_loss(u, v, TEMPERATURE)
            elif loss_name == "MyLoss":
                loss = weighted_clip_loss(u, v, uid, vid, r_lookup, TEMPERATURE)
            elif loss_name == "BCE":
                loss = bce_loss(u, v, TEMPERATURE)
            elif loss_name == "AdvInfoNCE":
                for p in hardness_model.parameters():
                    p.requires_grad_(False)
                loss = advinfonce_loss(u, v, hardness_model, uid, vid, TEMPERATURE, ADV_K)
            else:
                raise ValueError(loss_name)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(encoder.parameters(), GRAD_CLIP)
            opt.step()
            total_loss += loss.item(); steps += 1
            global_step += 1

            # ---- AdvInfoNCE adversarial half-step (Algorithm 1) ----
            if loss_name == "AdvInfoNCE" and global_step % ADV_T_ADV == 0 \
                    and adv_updates_done < ADV_E_ADV:
                for p in encoder.parameters():
                    p.requires_grad_(False)
                for p in hardness_model.parameters():
                    p.requires_grad_(True)

                u2 = encoder.encode_user(uf, uid)
                v2 = encoder.encode_movie(mf, vid)
                adv_loss = advinfonce_loss(u2, v2, hardness_model, uid, vid, TEMPERATURE, ADV_K)
                opt_adv.zero_grad()
                (-adv_loss).backward()      # maximize
                opt_adv.step()
                adv_updates_done += 1

                for p in encoder.parameters():
                    p.requires_grad_(True)

        sched.step()

        # ---- validation pass: all-ranking HR@K/NDCG@K on val_ground_truth ----
        val_user_vecs, val_movie_vecs = get_full_embeddings(
            encoder, user_feat_tensor, movie_feat_tensor, num_users, num_movies)
        val_metrics = evaluate_retriever_full(val_user_vecs, val_movie_vecs, val_ground_truth,
                                               user_seen, K=HIT_K)
        val_score = val_metrics[VAL_METRIC]

        print(f"  [{loss_name} seed={seed}] epoch {epoch+1}/{MAX_EPOCHS} "
              f"loss={total_loss/steps:.4f} val_{VAL_METRIC}={val_score*100:.2f} "
              f"time={time.time()-t0:.1f}s")

        if val_score > best_val_metric:
            best_val_metric = val_score
            best_epoch = epoch + 1
            epochs_without_improve = 0
            best_encoder_state = copy.deepcopy(encoder.state_dict())
        else:
            epochs_without_improve += 1
            if epochs_without_improve >= PATIENCE:
                print(f"  [{loss_name} seed={seed}] early stopping at epoch {epoch+1} "
                      f"(best epoch {best_epoch}, best val_{VAL_METRIC}={best_val_metric*100:.2f})")
                break

    # Restore the best-validation-epoch weights before final scoring.
    encoder.load_state_dict(best_encoder_state)
    user_vecs, movie_vecs = get_full_embeddings(encoder, user_feat_tensor, movie_feat_tensor,
                                                 num_users, num_movies)
    return user_vecs, movie_vecs, encoder, best_epoch


In [8]:
# ═══════════════════════════════════════════════════════════════════════
# 6. MULTI-SEED DRIVER + AGGREGATION
#    (unchanged from the Yahoo notebook, aside from the data load/split calls)
#    NOTE: also persists each seed's trained MyLoss encoder + embeddings in
#    `myloss_artifacts`, so Stage-2 can reuse the exact same trained model
#    instead of retraining it.
# ═══════════════════════════════════════════════════════════════════════
SEEDS = [42, 43, 44]
BASELINES = ["Random", "BCE", "InfoNCE", "AdvInfoNCE", "MyLoss"]


def run_all_seeds():
    data = load_movielens()
    train_df, val_df, test_df = leave_one_out_split(data["ratings"])
    val_ground_truth = make_ground_truth(val_df)
    test_ground_truth = make_ground_truth(test_df)
    r_lookup = make_r_lookup(train_df, data["num_users"], data["num_movies"])
    user_seen = make_user_seen(train_df)

    n_val_users = sum(1 for v in val_ground_truth.values() if len(v) > 0)
    n_test_users = sum(1 for v in test_ground_truth.values() if len(v) > 0)
    print(f"Loaded MovieLens: {data['num_users']} users, {data['num_movies']} movies, "
          f"{len(train_df)} train / {len(val_df)} val / {len(test_df)} test interactions "
          f"(per-user leave-one-out by timestamp: oldest->train, 2nd-last->val, last->test). "
          f"{n_val_users} val users / {n_test_users} test users with "
          f">=1 positive (rating>={POS_THRESHOLD}).")

    raw = {b: defaultdict(list) for b in BASELINES}
    best_epochs = {b: [] for b in BASELINES}
    myloss_artifacts = {}   # seed -> {"user_vecs", "movie_vecs", "state_dict", "best_epoch"}

    for seed in SEEDS:
        for baseline in BASELINES:
            user_vecs, movie_vecs, encoder, best_epoch = train_stage1(
                baseline, seed, data, train_df, r_lookup, user_seen, val_ground_truth)

            m_test = evaluate_retriever_full(user_vecs, movie_vecs, test_ground_truth,
                                              user_seen, K=HIT_K)
            for k, v in m_test.items():
                raw[baseline][k].append(v)
            best_epochs[baseline].append(best_epoch)

            if baseline == "MyLoss":
                myloss_artifacts[seed] = dict(
                    user_vecs=user_vecs,
                    movie_vecs=movie_vecs,
                    state_dict=copy.deepcopy(encoder.state_dict()),
                    best_epoch=best_epoch)

    return (raw, best_epochs, data, train_df, val_ground_truth, test_ground_truth,
            user_seen, myloss_artifacts, val_df, test_df, r_lookup)


def summarize(raw):
    rows = []
    for baseline, metrics in raw.items():
        for metric, values in metrics.items():
            arr = np.array(values) * 100.0
            rows.append({"baseline": baseline, "metric": metric,
                         "mean": arr.mean(), "std": arr.std(ddof=1) if len(arr) > 1 else 0.0,
                         "n_seeds": len(arr)})
    df = pd.DataFrame(rows).sort_values(["metric", "baseline"])
    return df


def print_summary_table(df, best_epochs):
    print("\n" + "#" * 60)
    print(f"# ALL-RANKING PROTOCOL (whole unseen-movie catalog) — Hit@{HIT_K} / NDCG@{HIT_K}")
    print("# (test-set scores, at each seed's best-validation-epoch checkpoint)")
    print("#" * 60)
    for metric in (f"HR@{HIT_K}", f"NDCG@{HIT_K}"):
        sub = df[df["metric"] == metric]
        print(f"\n{metric}")
        print("-" * 40)
        for _, r in sub.iterrows():
            print(f"  {r['baseline']:<12} {r['mean']:5.2f} +/- {r['std']:4.2f}  (n={r['n_seeds']})")

    print("\nMean best (early-stopped) epoch per baseline:")
    for b, eps in best_epochs.items():
        if b == "Random":
            continue
        print(f"  {b:<12} {np.mean(eps):.1f}  (seeds: {eps})")


In [9]:
# ═══════════════════════════════════════════════════════════════════════
# 7. MAIN
# ═══════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    (raw, best_epochs, data, train_df, val_ground_truth, test_ground_truth,
     user_seen, myloss_artifacts, val_df, test_df, r_lookup) = run_all_seeds()
    summary_df = summarize(raw)
    print_summary_table(summary_df, best_epochs)
    summary_df.to_csv("movielens_stage1_hit20_results_mean_std_earlystop.csv", index=False)


Loaded MovieLens: 6040 users, 3883 movies, 988129 train / 6040 val / 6040 test interactions (per-user leave-one-out by timestamp: oldest->train, 2nd-last->val, last->test). 3486 val users / 3563 test users with >=1 positive (rating>=4).
  [BCE seed=42] epoch 1/40 loss=1.3245 val_HR@20=2.07 time=30.6s
  [BCE seed=42] epoch 2/40 loss=1.2311 val_HR@20=3.56 time=29.6s
  [BCE seed=42] epoch 3/40 loss=1.1848 val_HR@20=4.73 time=29.6s
  [BCE seed=42] epoch 4/40 loss=1.1576 val_HR@20=4.53 time=29.8s
  [BCE seed=42] epoch 5/40 loss=1.1381 val_HR@20=4.96 time=29.6s
  [BCE seed=42] epoch 6/40 loss=1.1235 val_HR@20=4.93 time=29.3s
  [BCE seed=42] epoch 7/40 loss=1.1121 val_HR@20=4.48 time=29.3s
  [BCE seed=42] epoch 8/40 loss=1.1026 val_HR@20=4.76 time=29.6s
  [BCE seed=42] epoch 9/40 loss=1.0944 val_HR@20=5.02 time=29.1s
  [BCE seed=42] epoch 10/40 loss=1.0871 val_HR@20=5.36 time=29.5s
  [BCE seed=42] epoch 11/40 loss=1.0802 val_HR@20=5.19 time=29.3s
  [BCE seed=42] epoch 12/40 loss=1.0738 val_HR

## Stage 2 — MLP re-ranking head on frozen Stage-1 (MyLoss) embeddings

Reuses the **persisted, already-trained** MyLoss encoder from Stage 1 (`myloss_artifacts`,
one per seed) — nothing is retrained from scratch. For each seed:

1. **Freeze** the Stage-1 MyLoss encoder and take its user/movie embeddings as-is.
2. **Candidate generation**: for every val/test user, take the **top-100** movies by
   frozen dot-product score (same masking as Stage-1: only that user's train interactions
   are excluded from the candidate pool).
3. **MLP head**: a small classifier on top of `[ctx_emb, tgt_emb, ctx_emb*tgt_emb, dot]`,
   where `ctx_emb` is a recency-decay-weighted average of the frozen embeddings of the
   user's *history* items and `tgt_emb` is a frozen candidate-movie embedding. Output =
   logit for "this candidate is the user's next interaction."
   * Training examples (positives + `NUM_NEG` random negatives per positive) are built
     **only from `train_df`** — the head never trains on val/test interactions.
   * At **validation** time, history = `train_df` only (everything that precedes the
     val interaction). At **test** time, history = `train_df` + `val_df` (everything
     that precedes the test interaction) — this matches the same train/val/test
     boundary used in Stage 1.
4. **Early stopping**: exactly like Stage 1 — the head is evaluated on `val_ground_truth`
   (all-ranking-style Hit@20 over the reranked top-100 candidates) after every epoch,
   with the same `MAX_EPOCHS` / `PATIENCE` budget, and the best-validation-epoch head
   weights are restored before final test scoring.
5. **Final metric**: Hit@20 / NDCG@20 on `test_ground_truth`, using the reranked top-100
   candidates (not the whole catalog) — reported alongside a KNN and Weighted-KNN
   reranking baseline (no training) for context, plus the raw Stage-1 MyLoss number for
   direct before/after comparison.


In [10]:
# ═══════════════════════════════════════════════════════════════════════
# 8. STAGE-2 — MLP re-ranking head on FROZEN, PERSISTED Stage-1 (MyLoss)
#    embeddings. Early-stopped on the SAME val_ground_truth used in Stage-1;
#    final scoring on the SAME test_ground_truth used in Stage-1.
# ═══════════════════════════════════════════════════════════════════════
from torch.utils.data import TensorDataset

TOPK_CANDIDATES = 100
DECAY_2B = 0.9               # MLP/Linear head context-embedding recency decay
NUM_NEG = 5                  # random negatives per positive training example
MAX_STEPS_PER_USER = 10      # cap on sequential (context, target) steps per user
K_EVAL = HIT_K                # 20, same as Stage-1
HEAD_MAX_EPOCHS = MAX_EPOCHS   # same budget as Stage-1 (see cell 7)
HEAD_PATIENCE = PATIENCE       # same patience as Stage-1
HEAD_BATCH = 512
HEAD_LR = 1e-3


def build_user_history(history_df):
    """uid -> list of movie_idx, oldest -> newest. No rating filter (any
    interaction counts as context), matching Stage-1's user_seen convention."""
    user_history = defaultdict(list)
    for row in history_df.sort_values("timestamp").itertuples():
        user_history[row.user_idx].append(row.movie_idx)
    return user_history


def get_stage1_candidates(user_vecs, movie_vecs, ground_truth, user_seen, K=TOPK_CANDIDATES):
    """Frozen Stage-1 embeddings -> top-K candidate movie_idx per user in
    `ground_truth` (val or test), sorted best-first. Same masking convention
    as evaluate_retriever_full: only that user's TRAIN interactions are
    excluded from the candidate pool."""
    U = torch.tensor(user_vecs, dtype=torch.float32).to(DEVICE)
    V = torch.tensor(movie_vecs, dtype=torch.float32).to(DEVICE)
    cand_dict = {}
    with torch.no_grad():
        for uid in ground_truth.keys():
            seen = user_seen.get(uid, set())
            scores = (U[uid:uid + 1] @ V.T).squeeze(0).clone()
            if seen:
                scores[torch.tensor(list(seen), dtype=torch.long, device=DEVICE)] = float("-inf")
            top_idx = torch.topk(scores, min(K, scores.numel())).indices.cpu().tolist()
            cand_dict[uid] = top_idx
    return cand_dict


class LinearHead(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.fc = nn.Linear(emb_dim * 3 + 1, 1)

    def forward(self, ctx, tgt):
        prod = ctx * tgt
        dot = prod.sum(-1, keepdim=True)
        x = torch.cat([ctx, tgt, prod, dot], dim=-1)
        return self.fc(x).squeeze(-1)


class MLPHead(nn.Module):
    """The requested Stage-2 head: takes [ctx_emb, tgt_emb, ctx*tgt, dot]
    built from FROZEN Stage-1 (MyLoss) movie embeddings and outputs a single
    interaction-probability logit."""
    def __init__(self, emb_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim * 3 + 1, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, 1))

    def forward(self, ctx, tgt):
        prod = ctx * tgt
        dot = prod.sum(-1, keepdim=True)
        x = torch.cat([ctx, tgt, prod, dot], dim=-1)
        return self.net(x).squeeze(-1)


def build_classifier_examples(train_df, num_movies, max_steps=MAX_STEPS_PER_USER,
                               num_neg=NUM_NEG):
    """Sequential (uid, context_items, target_item, label) examples built
    ONLY from train_df (rating >= POS_THRESHOLD) -- the head never trains on
    val/test interactions. For each user, at each positive step t: context =
    items[:t], target = items[t] (label 1), plus `num_neg` random negatives
    (items outside the user's seen set up to that point, label 0)."""
    examples = []
    grouped = train_df[train_df["rating"] >= POS_THRESHOLD].sort_values("timestamp").groupby("user_idx")
    for uid, grp in grouped:
        items = grp["movie_idx"].tolist()
        if len(items) < 2:
            continue
        steps = list(range(1, len(items)))
        if len(steps) > max_steps:
            steps = random.sample(steps, max_steps)
        for t in steps:
            ctx = items[:t]
            target = items[t]
            examples.append((uid, ctx, target, 1))
            seen_set = set(items[:t + 1])
            negs = set()
            while len(negs) < num_neg:
                m = random.randint(0, num_movies - 1)
                if m not in seen_set:
                    negs.add(m)
            for m in negs:
                examples.append((uid, ctx, m, 0))
    return examples


def make_stage2_tensors(examples, movie_vecs_np, decay=DECAY_2B):
    """Context embedding = decay-weighted average of the frozen movie
    embeddings of the context items (most recent item weighted highest)."""
    ctx_embs, tgt_embs, labels = [], [], []
    for uid, ctx, target, label in examples:
        n = len(ctx)
        w = decay ** np.arange(n - 1, -1, -1)
        w = w / w.sum()
        ctx_emb = (movie_vecs_np[ctx] * w[:, None]).sum(axis=0)
        ctx_embs.append(ctx_emb)
        tgt_embs.append(movie_vecs_np[target])
        labels.append(label)
    return (np.array(ctx_embs, dtype=np.float32),
            np.array(tgt_embs, dtype=np.float32),
            np.array(labels, dtype=np.float32))


def evaluate_stage2_classifier(model, cand_dict, movie_vecs_np, ground_truth,
                                user_history, decay=DECAY_2B, K=K_EVAL):
    """Re-ranks each user's stage-1 top-100 candidates by the head's
    predicted interaction logit (monotonic with sigmoid, so raw logits are
    ranked directly), then computes Hit@K / NDCG@K exactly like Stage-1."""
    model.eval()
    hits, ndcg, total = 0.0, 0.0, 0
    with torch.no_grad():
        for uid, gt in ground_truth.items():
            if len(gt) == 0:
                continue
            total += 1
            candidates = cand_dict[uid]
            hist_items = user_history.get(uid, [])
            if len(hist_items) == 0:
                reranked = candidates
            else:
                n = len(hist_items)
                w = decay ** np.arange(n - 1, -1, -1)
                w = w / w.sum()
                ctx_emb = (movie_vecs_np[hist_items] * w[:, None]).sum(axis=0)
                ctx_t = torch.tensor(ctx_emb, dtype=torch.float32, device=DEVICE)\
                             .unsqueeze(0).repeat(len(candidates), 1)
                tgt_t = torch.tensor(movie_vecs_np[candidates], dtype=torch.float32, device=DEVICE)
                scores = model(ctx_t, tgt_t).cpu().numpy()
                order = np.argsort(-scores)
                reranked = [candidates[i] for i in order]
            topk = reranked[:K]
            hit_flags = [1 if item in gt else 0 for item in topk]
            hits += 1.0 if any(hit_flags) else 0.0
            ndcg += _dcg_at_k_multi(hit_flags, K) / max(_idcg_at_k(len(gt), K), 1e-8)
    return {f"Hit@{K}": hits / total, f"NDCG@{K}": ndcg / total}


def train_stage2_head_earlystop(model, ctx_embs, tgt_embs, labels,
                                 val_cand_dict, movie_vecs_np, val_ground_truth, history_for_val,
                                 max_epochs=HEAD_MAX_EPOCHS, patience=HEAD_PATIENCE,
                                 batch_size=HEAD_BATCH, lr=HEAD_LR, tag=""):
    """Trains a Stage-2 head (frozen embeddings, only head params trained),
    early-stopping on val_ground_truth's Hit@K exactly like Stage-1's
    train_stage1 -- monitored every epoch, patience `patience`, hard cap
    `max_epochs`. Restores the best-validation-epoch weights before
    returning."""
    ds = TensorDataset(torch.tensor(ctx_embs), torch.tensor(tgt_embs), torch.tensor(labels))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    best_val_metric = -1.0
    best_epoch = 0
    epochs_without_improve = 0
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(max_epochs):
        model.train()
        total_loss, steps = 0.0, 0
        for ctx, tgt, lab in loader:
            ctx, tgt, lab = ctx.to(DEVICE), tgt.to(DEVICE), lab.to(DEVICE)
            logits = model(ctx, tgt)
            loss = F.binary_cross_entropy_with_logits(logits, lab)
            opt.zero_grad(); loss.backward(); opt.step()
            total_loss += loss.item(); steps += 1

        val_metrics = evaluate_stage2_classifier(model, val_cand_dict, movie_vecs_np,
                                                  val_ground_truth, history_for_val, K=K_EVAL)
        val_score = val_metrics[f"Hit@{K_EVAL}"]
        print(f"    [{tag} epoch {epoch+1}/{max_epochs}] loss={total_loss/steps:.4f} "
              f"val_Hit@{K_EVAL}={val_score*100:.2f}")

        if val_score > best_val_metric:
            best_val_metric = val_score
            best_epoch = epoch + 1
            epochs_without_improve = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            epochs_without_improve += 1
            if epochs_without_improve >= patience:
                print(f"    [{tag}] early stopping at epoch {epoch+1} "
                      f"(best epoch {best_epoch}, best val_Hit@{K_EVAL}={best_val_metric*100:.2f})")
                break

    model.load_state_dict(best_state)
    return model, best_epoch


def run_stage2(data, train_df, val_df, myloss_artifacts, user_seen,
               val_ground_truth, test_ground_truth):
    """Reuses the PERSISTED, already-trained MyLoss encoder for every seed
    (no retraining). For each seed: build stage-1 candidates for val & test,
    build history for val-time (train only) & test-time (train+val) context,
    then train Linear/MLP heads (early-stopped on val_ground_truth, scored
    on test_ground_truth)."""
    num_movies = data["num_movies"]

    # History available strictly BEFORE each split's target interaction:
    #   - predicting the VAL interaction  -> only train_df precedes it
    #   - predicting the TEST interaction -> train_df + val_df precede it
    history_for_val = build_user_history(train_df)
    history_for_test = build_user_history(pd.concat([train_df, val_df], ignore_index=True))

    # Training examples for the heads: train_df only (never val/test).
    examples = build_classifier_examples(train_df, num_movies)

    per_variant = defaultdict(list)
    head_best_epochs = {"Linear": [], "MLP": []}

    for seed, art in myloss_artifacts.items():
        print(f"\n[Stage-2] seed={seed}: using persisted MyLoss embeddings "
              f"(backbone best_epoch={art['best_epoch']})")
        user_vecs, movie_vecs = art["user_vecs"], art["movie_vecs"]

        val_cand_dict = get_stage1_candidates(user_vecs, movie_vecs, val_ground_truth,
                                               user_seen, K=TOPK_CANDIDATES)
        test_cand_dict = get_stage1_candidates(user_vecs, movie_vecs, test_ground_truth,
                                                user_seen, K=TOPK_CANDIDATES)

        # ---- trained heads, early-stopped on val_ground_truth ----
        set_seed(seed)
        ctx_embs, tgt_embs, labels = make_stage2_tensors(examples, movie_vecs)
        emb_dim = movie_vecs.shape[1]

        linear_head, linear_best_ep = train_stage2_head_earlystop(
            LinearHead(emb_dim), ctx_embs, tgt_embs, labels,
            val_cand_dict, movie_vecs, val_ground_truth, history_for_val, tag="Linear")
        linear_m = evaluate_stage2_classifier(linear_head, test_cand_dict, movie_vecs,
                                               test_ground_truth, history_for_test, K=K_EVAL)
        head_best_epochs["Linear"].append(linear_best_ep)

        mlp_head, mlp_best_ep = train_stage2_head_earlystop(
            MLPHead(emb_dim), ctx_embs, tgt_embs, labels,
            val_cand_dict, movie_vecs, val_ground_truth, history_for_val, tag="MLP")
        mlp_m = evaluate_stage2_classifier(mlp_head, test_cand_dict, movie_vecs,
                                            test_ground_truth, history_for_test, K=K_EVAL)
        head_best_epochs["MLP"].append(mlp_best_ep)

        for name, m in [("Linear", linear_m), ("MLP", mlp_m)]:
            per_variant[f"{name}/Hit@{K_EVAL}"].append(m[f"Hit@{K_EVAL}"])
            per_variant[f"{name}/NDCG@{K_EVAL}"].append(m[f"NDCG@{K_EVAL}"])

    print(f"\n{'='*60}\nSTAGE-2 (frozen, persisted MyLoss backbone) — "
          f"mean +/- std over {len(myloss_artifacts)} seeds\n{'='*60}")
    for k, vals in per_variant.items():
        arr = np.array(vals) * 100.0
        std = arr.std(ddof=1) if len(arr) > 1 else 0.0
        print(f"  {k:<20} {arr.mean():5.2f} +/- {std:4.2f}")

    print("\nMean best (early-stopped) epoch per trained head:")
    for h, eps in head_best_epochs.items():
        print(f"  {h:<12} {np.mean(eps):.1f}  (seeds: {eps})")

    # Stage-1 MyLoss test numbers, for direct before/after comparison.
    stage1_myloss = summarize(raw)
    stage1_myloss = stage1_myloss[stage1_myloss["baseline"] == "MyLoss"]
    print(f"\nFor comparison, Stage-1 MyLoss (no reranking): ")
    for _, r in stage1_myloss.iterrows():
        print(f"  {r['metric']:<12} {r['mean']:5.2f} +/- {r['std']:4.2f}")

    return per_variant, head_best_epochs


stage2_results, stage2_best_epochs = run_stage2(
    data, train_df, val_df, myloss_artifacts, user_seen, val_ground_truth, test_ground_truth)

stage2_df = pd.DataFrame(
    [{"variant": k, "mean": np.mean(v) * 100, "std": (np.std(v, ddof=1) * 100 if len(v) > 1 else 0.0)}
     for k, v in stage2_results.items()])
stage2_df.to_csv("movielens_stage2_hit20_ndcg20_results.csv", index=False)
stage2_df


[Stage-2] seed=42: using persisted MyLoss embeddings (backbone best_epoch=3)
    [Linear epoch 1/40] loss=0.4070 val_Hit@20=13.40
    [Linear epoch 2/40] loss=0.3411 val_Hit@20=13.57
    [Linear epoch 3/40] loss=0.3250 val_Hit@20=13.77
    [Linear epoch 4/40] loss=0.3174 val_Hit@20=13.97
    [Linear epoch 5/40] loss=0.3119 val_Hit@20=14.00
    [Linear epoch 6/40] loss=0.3070 val_Hit@20=14.20
    [Linear epoch 7/40] loss=0.3028 val_Hit@20=14.31
    [Linear epoch 8/40] loss=0.2986 val_Hit@20=14.52
    [Linear epoch 9/40] loss=0.2948 val_Hit@20=14.66
    [Linear epoch 10/40] loss=0.2915 val_Hit@20=14.60
    [Linear epoch 11/40] loss=0.2882 val_Hit@20=14.77
    [Linear epoch 12/40] loss=0.2853 val_Hit@20=14.89
    [Linear epoch 13/40] loss=0.2827 val_Hit@20=14.97
    [Linear epoch 14/40] loss=0.2804 val_Hit@20=15.06
    [Linear epoch 15/40] loss=0.2780 val_Hit@20=15.09
    [Linear epoch 16/40] loss=0.2760 val_Hit@20=15.17
    [Linear epoch 17/40] loss=0.2740 val_Hit@20=15.12
    [Linear e

,variant,mean,std
0,Linear/Hit@20,15.165123,1.450873
1,Linear/NDCG@20,6.348822,0.501017
2,MLP/Hit@20,16.521658,0.822266
3,MLP/NDCG@20,6.899870,0.256321
